# Day 031 — Exercise 5: ai_resilience_report

**What you'll build:** `ai_resilience_report(batch_results, dlq_items, model='llama3.2') -> str` — formats batch run statistics and DLQ contents into a text block, then asks Ollama to produce a 2–3 sentence incident report with a recommended action.

**Why it matters:** A list of step result dicts is useful for machines. An incident report is useful for engineers. ai_resilience_report bridges the gap: it translates failure statistics into actionable natural language.

In [ ]:
import ollama
import time
from datetime import datetime

## Provided: All Helper Functions

In [ ]:
import time


def retry(fn, max_attempts: int = 3, base_delay: float = 1.0, backoff: float = 2.0):
    last_error = None
    for attempt in range(max_attempts):
        try:
            return fn()
        except Exception as e:
            last_error = e
            if attempt < max_attempts - 1:
                time.sleep(base_delay * (backoff ** attempt))
    raise last_error


from datetime import datetime


class DeadLetterQueue:
    def __init__(self):
        self._items: list = []

    def add(self, item, error: str, context: dict | None = None) -> None:
        self._items.append({
            "item":     item,
            "error":    error,
            "context":  context or {},
            "added_at": datetime.now().isoformat(),
        })

    def drain(self) -> list:
        items, self._items = self._items, []
        return items

    def peek(self) -> list:
        return list(self._items)

    def size(self) -> int:
        return len(self._items)


def resilient_step(
    name: str,
    fn,
    max_attempts: int = 3,
    base_delay: float = 1.0,
    backoff: float = 2.0,
) -> dict:
    start      = time.time()
    last_error = None
    for attempt in range(max_attempts):
        try:
            result = fn()
            return {
                "name":       name,
                "status":     "ok",
                "result":     result,
                "error":      None,
                "duration_s": round(time.time() - start, 3),
                "attempts":   attempt + 1,
            }
        except Exception as e:
            last_error = e
            if attempt < max_attempts - 1:
                time.sleep(base_delay * (backoff ** attempt))
    return {
        "name":       name,
        "status":     "error",
        "result":     None,
        "error":      str(last_error),
        "duration_s": round(time.time() - start, 3),
        "attempts":   max_attempts,
    }


def process_batch_with_dlq(
    items: list,
    process_fn,
    dlq: DeadLetterQueue,
    max_attempts: int = 3,
    base_delay: float = 1.0,
    backoff: float = 2.0,
) -> list:
    results = []
    for item in items:
        r = resilient_step(
            str(item),
            lambda i=item: process_fn(i),
            max_attempts=max_attempts,
            base_delay=base_delay,
            backoff=backoff,
        )
        if r["status"] == "error":
            dlq.add(item, r["error"])
        results.append(r)
    return results

## Your Implementation

In [ ]:
def ai_resilience_report(
    batch_results: list,
    dlq_items: list,
    model: str = 'llama3.2',
) -> str:
    """
    Return a 2–3 sentence incident report with recommended action.

    Args:
        batch_results: List of step result dicts from process_batch_with_dlq.
        dlq_items:     List of DLQ records (each has item, error, added_at).
        model:         Ollama model name.

    Returns:
        Concise natural-language report as a string.
    """
    # TODO: total=len(batch_results); passed=sum(status=='ok')
    # TODO: avg_attempts = round(sum(r.get('attempts',1) for r in batch_results)/total, 2) if total else 0.0
    # TODO: lines = [f'Batch run: {passed}/{total} items succeeded, {len(dlq_items)} failed to DLQ.',
    #                 f'Average attempts per item: {avg_attempts}.']
    # TODO: for entry in dlq_items[:3]: lines.append(f'  DLQ: {entry["item"]} \u2014 {entry["error"]}')
    # TODO: ollama.chat with reliability-engineer system prompt
    # TODO: return response['message']['content']
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    def _sr(name, status, attempts=1):
        return {'name': name, 'status': status, 'result': None,
                'error': None if status == 'ok' else 'err', 'duration_s': 0.1, 'attempts': attempts}

    ALL_OK = [_sr('a', 'ok'), _sr('b', 'ok'), _sr('c', 'ok', 2)]
    EMPTY_DLQ = []

    MIXED = [_sr('a', 'ok'), _sr('b', 'error', 3), _sr('c', 'ok'), _sr('d', 'error', 3)]
    FULL_DLQ = [
        {'item': 'url_1', 'error': 'ConnectionTimeout', 'added_at': '2026-01-01T00:00:00'},
        {'item': 'url_2', 'error': 'ReadTimeout',       'added_at': '2026-01-01T00:00:01'},
    ]

    # Check 1: defined
    try:
        assert 'ai_resilience_report' in globals()
        passed += 1; print('\u2705 Check 1: ai_resilience_report defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    ok_report = None

    # Check 2: returns a string for all-ok input
    try:
        ok_report = ai_resilience_report(ALL_OK, EMPTY_DLQ)
        assert isinstance(ok_report, str), \
            f'expected str, got {type(ok_report)}'
        passed += 1; print('\u2705 Check 2: returns a string for all-ok input')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: non-empty report
    try:
        assert ok_report is not None
        assert len(ok_report.strip()) > 10, \
            f'report too short: {ok_report!r}'
        passed += 1; print(f'\u2705 Check 3: report is {len(ok_report)} chars')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: works with mixed (failures + DLQ items)
    try:
        err_report = ai_resilience_report(MIXED, FULL_DLQ)
        assert isinstance(err_report, str) and len(err_report) > 10
        passed += 1; print('\u2705 Check 4: works with mixed batch + DLQ items')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: empty batch does not crash
    try:
        empty_report = ai_resilience_report([], [])
        assert isinstance(empty_report, str) and len(empty_report) > 5
        passed += 1; print('\u2705 Check 5: empty batch returns a string without error')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def ai_resilience_report(
    batch_results: list,
    dlq_items: list,
    model: str = "llama3.2",
) -> str:
    total        = len(batch_results)
    passed       = sum(1 for r in batch_results if r["status"] == "ok")
    avg_attempts = (
        round(sum(r.get("attempts", 1) for r in batch_results) / total, 2)
        if total else 0.0
    )
    lines = [
        f"Batch run: {passed}/{total} items succeeded, "
        f"{len(dlq_items)} failed to DLQ.",
        f"Average attempts per item: {avg_attempts}.",
    ]
    for entry in dlq_items[:3]:
        lines.append(f"  DLQ: {entry['item']} \u2014 {entry['error']}")

    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a reliability engineer. "
                    "Summarise a batch processing run in 2\u20133 sentences. "
                    "Focus on the failure rate and recommend one concrete action."
                ),
            },
            {
                "role": "user",
                "content": "\n".join(lines) + "\n\nSummarise and recommend:",
            },
        ],
    )
    return response["message"]["content"]
```

</details>